# Trading APP


This notebook now runs the full process end to end:

1. Train the traditional ML live-trade stack.
2. Save the classifier, regressor, and autoencoder artifacts.
3. Build and save the latest leaderboard.
4. Build and save historical entry-trade embeddings for similar-trade search.
5. Run the similar-trades lookup for a chosen symbol using those saved artifacts.
6. Launch the Streamlit trading app for the app UI.


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

_repo_start = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in (_repo_start, *_repo_start.parents) if (candidate / "app").is_dir() and (candidate / "notebooks").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError(f"Could not locate repo root from cwd={_repo_start}")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from app.live_trade_leaderboard import run_live_trade_leaderboard_build
from app.optimal_trade_lookup import find_nearest_optimal_trades
from app.trading_notebook import (
    bootstrap_repo,
    live_trade_notebook_summary,
    load_recent_similarity_build,
    make_live_trade_notebook_config,
    make_similarity_query,
)
from app.notebook_runtime import NotebookProgress, StreamlitProcess

bootstrap_repo(repo_root)
progress = NotebookProgress()
_nb_log = progress.log
run_with_progress = progress.run
streamlit_process = StreamlitProcess(
    repo_root=repo_root,
    app_path=repo_root / "app" / "streamlit_optimal_trade_finder.py",
    logger=_nb_log,
)

_nb_log(f"Notebook bootstrap complete. Repo root: {repo_root}")
_nb_log(f"Working directory set to: {Path.cwd().resolve()}")
_nb_log(f"Python executable: {sys.executable}")
_nb_log(f"DJANGO_SETTINGS_MODULE={os.environ.get('DJANGO_SETTINGS_MODULE')}")


[17:55:29] Notebook bootstrap complete. Repo root: /home/jlee153232/PycharmProjects/optimal_trader
[17:55:29] Working directory set to: /home/jlee153232/PycharmProjects/optimal_trader
[17:55:29] Python executable: /home/jlee153232/miniconda3/envs/optimal_trader/bin/python
[17:55:29] DJANGO_SETTINGS_MODULE=settings


In [2]:
QUERY_SYMBOL = "AAPL"
SIMILAR_TRADES_TOP_K = 10
LEADERBOARD_TOP_K = 20
UNIVERSE_SOURCE = "auto"  # auto, screener, local
EXPLICIT_SYMBOLS = []  # Example: ["AAPL", "MSFT"] or "AAPL,MSFT"
DATA_START = "1900-01-01"
MIN_MARKET_CAP = 10_000_000_000.0
REFRESH_FMP_DATA = True
SKIP_CACHED_INACTIVE_SYMBOLS = True
REFRESH_MACRO_DATA = True

APP_CFG = make_live_trade_notebook_config(
    repo_root,
    data_start=DATA_START,
    min_market_cap=MIN_MARKET_CAP,
    refresh_fmp_data=REFRESH_FMP_DATA,
    skip_cached_inactive_symbols=SKIP_CACHED_INACTIVE_SYMBOLS,
    refresh_macro_data=REFRESH_MACRO_DATA,
    leaderboard_top_k=LEADERBOARD_TOP_K,
    universe_source=UNIVERSE_SOURCE,
    symbols=EXPLICIT_SYMBOLS,
)
_nb_log("Configuration prepared for end-to-end run.")
_nb_log(f"Artifact dir: {APP_CFG['runtime']['artifact_dir']}")
_nb_log(f"Universe min market cap: ${APP_CFG['universe']['min_market_cap']:,.0f}")
_nb_log(f"Download missing FMP symbol data: {APP_CFG['fmp_refresh']['refresh_symbol_sections_before_build']}")
_nb_log(f"Skip cached inactive symbols: {APP_CFG['fmp_refresh']['skip_cached_inactive_symbols']}")
_nb_log(f"Leaderboard top_k: {APP_CFG['strategy']['top_k']}")
_nb_log(f"Similar-trades query symbol: {QUERY_SYMBOL}")

display(pd.DataFrame([live_trade_notebook_summary(
    APP_CFG,
    query_symbol=QUERY_SYMBOL,
    similar_trades_top_k=SIMILAR_TRADES_TOP_K,
)]))


[17:55:29] Configuration prepared for end-to-end run.
[17:55:29] Artifact dir: /home/jlee153232/PycharmProjects/optimal_trader/artifacts/raw_stack
[17:55:29] Universe min market cap: $10,000,000,000
[17:55:29] Download missing FMP symbol data: True
[17:55:29] Skip cached inactive symbols: True
[17:55:29] Leaderboard top_k: 20
[17:55:29] Similar-trades query symbol: AAPL


,query_symbol,universe_source,explicit_symbol_count,data_start,data_end,min_market_cap,download_missing_fmp_data,skip_cached_inactive_symbols,refresh_macro_data,leaderboard_top_k,similar_trades_top_k,artifact_dir
0,AAPL,auto,0,1900-01-01,2026-06-10,1.000000e+10,True,True,True,20,10,/home/jlee153232/PycharmProjects/optimal_trade...


In [3]:
recent_build_result, artifact_status = load_recent_similarity_build(APP_CFG, max_age=pd.Timedelta(days=1))
model_source = "retrained"
if recent_build_result is not None:
    build_result = recent_build_result
    model_source = "artifacts"
    _nb_log(
        "Reusing saved raw-stack + leaderboard artifacts from "
        f"{artifact_status['artifact_dir']} | age {artifact_status['age'] / pd.Timedelta(hours=1):.2f}h"
    )
else:
    _nb_log(
        "Saved artifacts were not reusable; running full leaderboard build | "
        f"reason={artifact_status.get('reason') or 'unknown'}"
    )
    build_result = run_with_progress(
        "Train models, save artifacts, build leaderboard, and write trade vectors",
        run_live_trade_leaderboard_build,
        config=APP_CFG,
        progress_logger=_nb_log,
        heartbeat_seconds=0,
    )

_nb_log(f"Latest scored date: {pd.Timestamp(build_result.latest_date).date().isoformat()}")
_nb_log(f"Universe size: {len(build_result.universe):,}")
_nb_log(f"Historical entry trades embedded: {build_result.reference_trade_count:,}")
_nb_log(f"Vector backend: {str(build_result.vector_metadata.get('backend') or '')}")

leaderboard_summary = pd.DataFrame([
    {
        "latest_date": pd.Timestamp(build_result.latest_date).date().isoformat(),
        "universe_size": len(build_result.universe),
        "leaderboard_rows": len(build_result.leaderboard),
        "reference_trade_count": build_result.reference_trade_count,
        "artifact_dir": str(build_result.artifact_dir),
        "vector_backend": str(build_result.vector_metadata.get("backend") or ""),
        "model_source": model_source,
        "artifact_age_hours": None if pd.isna(artifact_status.get("age")) else round(float(artifact_status["age"] / pd.Timedelta(hours=1)), 2),
        "artifact_status": str(artifact_status.get("reason") or ""),
        "saved_latest_date": str(artifact_status.get("saved_latest_date") or ""),
        "expected_score_date": str(artifact_status.get("expected_score_date") or ""),
    }
])

display(leaderboard_summary)
display(build_result.leaderboard.head(25))


[17:55:29] Reusing saved raw-stack + leaderboard artifacts from /home/jlee153232/PycharmProjects/optimal_trader/artifacts/raw_stack | age 0.43h
[17:55:29] Latest scored date: 2026-06-09
[17:55:29] Universe size: 820
[17:55:29] Historical entry trades embedded: 226,933
[17:55:29] Vector backend: numpy


,latest_date,universe_size,leaderboard_rows,reference_trade_count,artifact_dir,vector_backend,model_source,artifact_age_hours,artifact_status,saved_latest_date,expected_score_date
0,2026-06-09,820,819,226933,/home/jlee153232/PycharmProjects/optimal_trade...,numpy,artifacts,0.43,fresh_saved_build,,


,Rank,Scored Date,Symbol,Direction,Eligible,Classifier Score,Regressor Score,Autoencoder Score,Combined Score,Similar Trades,__price,__long_score,__short_score,__score_col_name,__component__0,__component__1,__component__2,__component__3,__component__4,__component__5
0,1,2026-06-05,PSTG,Long,True,0.981591,0.697245,1.000000,0.912386,/?symbol=PSTG,72.17,0.912386,0.630977,buy_score_mean_raw_pct6,0.981591,0.697245,1.000000,0.863248,0.948718,0.983516
1,2,2026-06-09,CNC,Short,True,0.979831,0.766393,0.936638,0.904816,/?symbol=CNC,66.21,0.607713,0.904816,short_score_mean_raw_pct6,0.979831,0.766393,0.936638,0.912088,0.985348,0.848596
2,3,2026-06-09,GWW,Short,True,0.978006,0.612020,1.000000,0.896262,/?symbol=GWW,1329.80,0.601803,0.896262,short_score_mean_raw_pct6,0.978006,0.612020,1.000000,0.905983,0.887668,0.993895
3,4,2026-06-09,UNM,Short,True,0.991928,0.560159,1.000000,0.892991,/?symbol=UNM,88.00,0.567436,0.892991,short_score_mean_raw_pct6,0.991928,0.560159,1.000000,0.985348,0.826618,0.993895
4,5,2026-06-09,CEG,Long,True,0.995722,0.622446,0.951938,0.892942,/?symbol=CEG,251.65,0.892942,0.561645,buy_score_mean_raw_pct6,0.995722,0.622446,0.951938,0.998779,0.899878,0.888889
5,6,2026-06-09,COIN,Long,True,0.992736,0.775594,0.900375,0.885362,/?symbol=COIN,155.50,0.885362,0.569306,buy_score_mean_raw_pct6,0.992736,0.775594,0.900375,0.956044,0.987790,0.699634
6,7,2026-06-09,HUBS,Long,True,0.963745,0.645745,1.000000,0.883328,/?symbol=HUBS,197.69,0.883328,0.630659,buy_score_mean_raw_pct6,0.963745,0.645745,1.000000,0.794872,0.916972,0.978632
7,8,2026-06-09,VIAV,Long,True,0.979361,0.664306,0.953435,0.878923,/?symbol=VIAV,46.43,0.878923,0.602734,buy_score_mean_raw_pct6,0.979361,0.664306,0.953435,0.849817,0.934066,0.892552
8,9,2026-06-09,PS,Long,True,0.987383,0.696906,0.915609,0.871045,/?symbol=PS,33.34,0.871045,0.573053,buy_score_mean_raw_pct6,0.987383,0.696906,0.915609,0.907204,0.947497,0.771673
9,10,2026-06-09,DXCM,Short,True,0.986925,0.702271,0.906919,0.870822,/?symbol=DXCM,78.19,0.558330,0.870822,short_score_mean_raw_pct6,0.986925,0.702271,0.906919,0.951160,0.951160,0.726496


In [4]:
import os
print(bool(os.getenv("ROBINHOOD_USERNAME")), bool(os.getenv("ROBINHOOD_PASSWORD")))

True True


In [5]:
query = make_similarity_query(
    build_result,
    APP_CFG,
    query_symbol=QUERY_SYMBOL,
    top_k=SIMILAR_TRADES_TOP_K,
)
query


OptimalTradeQuery(symbol='AAPL', as_of_date='2026-06-09', query_lookback_years=0, reference_symbols=(), reference_start_date='1900-01-01', reference_end_date='2026-06-08', top_k=10, label_freq='YE', label_k_values=(1, 2, 4, 8), min_profit_pct_points=0.0, download_missing_prices=False, artifact_dir='/home/jlee153232/PycharmProjects/optimal_trader/artifacts/raw_stack', db_artifact_name='', db_artifact_version=None)

In [6]:
_nb_log(f"Running similar-trades search for {query.symbol} as of {query.as_of_date} with top_k={query.top_k}")
result = run_with_progress(
    f"Find nearest optimal trades for {query.symbol}",
    find_nearest_optimal_trades,
    query,
    heartbeat_seconds=15,
)

_nb_log(f"Nearest trades found: {len(result.nearest_trades):,}")
_nb_log(f"Search method: {result.metadata.get('search_method')}")
_nb_log(f"Lookup source: {result.metadata.get('embedding_source', result.metadata.get('trade_vector_backend', ''))}")

display(result.query_summary)
display(result.nearest_trades)
result.metadata

[17:55:29] Running similar-trades search for AAPL as of 2026-06-09 with top_k=10
[17:55:29] Starting: Find nearest optimal trades for AAPL
[17:55:33] Finished: Find nearest optimal trades for AAPL in 0.1 min
[17:55:33] Nearest trades found: 10
[17:55:33] Search method: prebuilt_trade_vector_index
[17:55:33] Lookup source: numpy


,symbol,as_of_date,ae_familiarity,artifact_source,artifact_numeric_feature_count,reference_trade_count
0,AAPL,2026-06-09,0.704039,/home/jlee153232/PycharmProjects/optimal_trade...,195,424


,Symbol,Side,Entry Date,Exit Date,Signed Trade Return,Hold Days,AE Familiarity,Entry Price,Exit Price
0,AMD,short,2025-11-12,2025-11-25,-20.18,13,1.000000,258.89,206.13
1,LRCX,long,2026-02-05,2026-02-24,14.31,19,0.780113,213.05,243.96
2,SMCI,short,2024-03-26,2024-04-22,-29.86,27,0.692470,102.51,71.70
3,LRCX,long,2025-11-21,2025-12-26,24.84,35,0.763136,142.24,177.86
4,AMD,long,2024-01-03,2024-03-06,55.45,63,0.773875,135.32,210.63
5,MU,long,2026-02-10,2026-02-25,14.74,15,0.612919,373.09,428.82
6,GLW,long,2026-04-29,2026-05-13,35.75,14,0.904085,151.90,206.51
7,SMCI,long,2024-02-21,2024-03-13,61.62,21,0.681342,73.42,118.81
8,GLW,long,2026-03-17,2026-05-13,58.79,57,0.856444,129.89,206.51
9,GLW,long,2026-03-09,2026-05-14,61.19,66,0.862506,129.05,208.28


{'artifact_source': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/raw_stack/ae_raw.pkl',
 'artifact_metadata': {'artifact_version': 1,
  'stack': 'raw',
  'feature_list': ['open',
   'high',
   'low',
   'close',
   'volume',
   'px__ret_1d',
   'px__ret_2d',
   'px__ret_3d',
   'px__ret_5d',
   'px__ret_10d',
   'px__ret_20d',
   'px__ret_21d',
   'px__ret_63d',
   'px__ret_126d',
   'px__ret_189d',
   'px__ret_252d',
   'px__dist_sma_5',
   'px__sma_slope_5',
   'px__dist_sma_10',
   'px__sma_slope_10',
   'px__dist_sma_20',
   'px__sma_slope_20',
   'px__dist_sma_50',
   'px__sma_slope_50',
   'px__dist_sma_100',
   'px__sma_slope_100',
   'px__dist_sma_200',
   'px__sma_slope_200',
   'px__dist_ema_12',
   'px__dist_ema_26',
   'px__dist_ema_50',
   'px__macd',
   'px__macd_signal',
   'px__macd_hist',
   'px__z_close_10',
   'px__bb_pos_10',
   'px__z_close_20',
   'px__bb_pos_20',
   'px__z_close_63',
   'px__bb_pos_63',
   'px__hl_range',
   'px__oc_change',
   'px_

## Open Streamlit Trading App

Start the Streamlit trading app and open the URL below. The Streamlit app includes the leaderboard, target sheet, and Robinhood option workflow.


In [7]:
streamlit_url = streamlit_process.start(port=8502, force_restart=True)
streamlit_urls = {"streamlit_app": streamlit_url, "leaderboard_page": streamlit_url}
streamlit_urls


[17:55:33] Starting Streamlit trading app: /home/jlee153232/PycharmProjects/optimal_trader/app/streamlit_optimal_trade_finder.py on port 8502


{'streamlit_app': 'http://localhost:8502',
 'leaderboard_page': 'http://localhost:8502'}

## Notes

- The leaderboard build reuses the traditional ML live-trade training path.
- It saves the raw-stack model artifacts into `artifacts/raw_stack`.
- It also saves historical entry-trade latent vectors so the similar-trades page can search faster.
- The similar-trades query uses those saved artifacts instead of assuming they already existed.